# ABIDE Dataset Exploration

The ABIDE (Autism Brain Imaging Data Exchange) dataset contains resting-state fMRI
and structural MRI from ~1000 subjects across 17 international sites.
Labels: **ASD** (Autism Spectrum Disorder) vs **TDC** (Typically Developing Controls).

This notebook mirrors `01_data_exploration.ipynb` but for ABIDE, validating that
the same preprocessing pipeline works before training.

In [ ]:
# Standard imports — same pipeline as ADHD-200.
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import Counter
from torch.utils.data import DataLoader

from utils.preprocessing import (
    load_phenotypic, build_subject_file_map,
    preprocess_volume, load_nifti, get_crop_shape
)
from utils.dataset import build_datasets

# Point to the ABIDE data directory (downloaded by download_abide.py).
DATA_DIR = '../data/raw_abide'
print('Imports OK')

## 1. Phenotypic CSV
Check subject counts and class balance. ABIDE has ~1000 subjects vs ADHD-200's 79.

In [ ]:
# Load the phenotypic CSV saved by download_abide.py.
# label=0 → ASD (Autism), label=1 → TDC (Control) — same convention as ADHD-200.
pheno = load_phenotypic(DATA_DIR)
print(f'Total subjects in CSV: {len(pheno)}')
print(f'  ASD (label=0): {(pheno["label"]==0).sum()}')
print(f'  TDC (label=1): {(pheno["label"]==1).sum()}')
print(f'Columns: {list(pheno.columns)}')
pheno.head()

## 2. Subject File Map
Check how many subjects have all 3 derivatives downloaded successfully.

In [ ]:
# Build a map of {subject_id: {derivative: path}} for subjects with downloaded files.
# Subjects missing any derivative are excluded from training.
file_map = build_subject_file_map(DATA_DIR)
print(f'Subjects with at least one derivative: {len(file_map)}')

complete = [sid for sid, d in file_map.items() if len(d) == 3]
falff_only = [sid for sid, d in file_map.items() if 'falff' in d]
gm_only    = [sid for sid, d in file_map.items() if 'gm' in d]

print(f'Subjects with all 3 derivatives: {len(complete)}')
print(f'Subjects with fALFF: {len(falff_only)}')
print(f'Subjects with GM:    {len(gm_only)}')

if complete:
    sid = complete[0]
    print(f'\nExample subject {sid}:')
    for k, v in file_map[sid].items():
        print(f'  {k}: {v}')

## 3. Raw Volume Shapes
ABIDE volumes may differ in shape from ADHD-200. Luis added zero-padding in
`preprocessing.py` to handle volumes smaller than the crop target.

In [ ]:
# Check raw shapes for the first complete subject.
# Target crop shapes: fMRI → (47,60,46), sMRI → (90,117,100).
# Volumes smaller than the target will be zero-padded before cropping.
if complete:
    sid = complete[0]
    for deriv, path in file_map[sid].items():
        vol = load_nifti(path)
        target = get_crop_shape(deriv)
        fits = all(v >= t for v, t in zip(vol.shape, target))
        print(f'{deriv:6s}: raw={vol.shape}  target={target}  '
              f'fits={fits}  min={vol.min():.3f}  max={vol.max():.3f}')

# Check across all subjects — count how many need padding.
print('\nChecking all subjects for padding requirement...')
needs_padding = []
for s in complete:
    for deriv, path in file_map[s].items():
        vol = load_nifti(path)
        target = get_crop_shape(deriv)
        if not all(v >= t for v, t in zip(vol.shape, target)):
            needs_padding.append((s, deriv, vol.shape))

print(f'  Subjects needing padding: {len(needs_padding)} / {len(complete)*3} volumes')
if needs_padding:
    for x in needs_padding[:5]:
        print(f'    {x}')

## 4. Normalization Check
Verify z-score normalization brings all volumes to zero mean and unit std.

In [ ]:
# Before normalization: raw intensity values vary widely across subjects and sites.
# After normalization: mean ≈ 0, std ≈ 1 for every volume.
if complete:
    sid = complete[0]
    for deriv, path in file_map[sid].items():
        raw = load_nifti(path)
        t   = preprocess_volume(path, deriv).squeeze().numpy()
        print(f'{deriv:6s}  raw: mean={raw.mean():.4f} std={raw.std():.4f} '
              f'  →  normed: mean={t.mean():.4f} std={t.std():.4f}')

## 5. Shape Check Across All Subjects
Every preprocessed volume must be exactly the crop target shape.

In [ ]:
# If any volume fails this check, training will crash with a shape mismatch error.
errors = []
for sid in complete:
    for deriv, path in file_map[sid].items():
        try:
            t = preprocess_volume(path, deriv)
            expected = get_crop_shape(deriv)
            if tuple(t.shape[1:]) != expected:
                errors.append((sid, deriv, f'shape mismatch: {tuple(t.shape)}'))
        except Exception as e:
            errors.append((sid, deriv, str(e)))

if errors:
    print(f'ERRORS ({len(errors)}):')
    for e in errors: print(' ', e)
else:
    print(f'All {len(complete)*3} volumes passed shape check.')

## 6. Class Distribution
Check ASD/TDC balance across the downloaded subjects.

In [ ]:
# A heavily imbalanced dataset can bias the model toward the majority class.
# ABIDE is roughly 50/50 but varies by site.
ds_falff = build_datasets(DATA_DIR, derivative='falff')
labels_all = [ds_falff[i][1] for i in range(len(ds_falff))]
counts = Counter(labels_all)

print(f'Subjects in fALFF dataset: {len(ds_falff)}')
print(f'  ASD (label=0): {counts[0]}  ({100*counts[0]/len(ds_falff):.1f}%)')
print(f'  TDC (label=1): {counts[1]}  ({100*counts[1]/len(ds_falff):.1f}%)')

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(['ASD', 'TDC'], [counts[0], counts[1]], color=['steelblue', 'salmon'])
ax.set_ylabel('Subjects')
ax.set_title('ABIDE Class Distribution')
plt.tight_layout()
plt.show()

## 7. Brain Slice Visualisation
Compare axial mid-slices of fALFF, ReHo, and GM for one ABIDE subject.

In [ ]:
# Visual sanity check — slices should look like brain maps, not noise or all zeros.
# fALFF/ReHo show functional activation patterns; GM shows grey matter density.
if complete:
    sid = complete[0]
    fig, axes = plt.subplots(1, len(file_map[sid]), figsize=(14, 4))
    for ax, (deriv, path) in zip(axes, file_map[sid].items()):
        t = preprocess_volume(path, deriv).squeeze().numpy()
        mid = t.shape[2] // 2
        ax.imshow(t[:, :, mid].T, cmap='gray', origin='lower')
        ax.set_title(f'{deriv}  (subject {sid})')
        ax.axis('off')
    plt.suptitle('ABIDE — mid axial slice (z-scored)', y=1.02)
    plt.tight_layout()
    plt.show()

## 8. Dataset Objects
Build PyTorch datasets for all three derivatives and verify shapes.

In [ ]:
# Build datasets for all three modalities.
# These are the objects that will be fed into the DataLoader during training.
ds_falff = build_datasets(DATA_DIR, derivative='falff')
ds_reho  = build_datasets(DATA_DIR, derivative='reho')
ds_gm    = build_datasets(DATA_DIR, derivative='gm')
ds_multi = build_datasets(DATA_DIR, multi_modal=True)

print(f'fALFF dataset : {len(ds_falff)} subjects')
print(f'ReHo  dataset : {len(ds_reho)} subjects')
print(f'GM    dataset : {len(ds_gm)} subjects')
print(f'Multi dataset : {len(ds_multi)} subjects')

tensor, label = ds_falff[0]
print(f'\nfALFF sample 0 — shape: {tuple(tensor.shape)}, label: {label} ({"ASD" if label==0 else "TDC"})')

fmri, smri, label = ds_multi[0]
print(f'Multi  sample 0 — fMRI: {tuple(fmri.shape)}, sMRI: {tuple(smri.shape)}, label: {label}')

## 9. Compare ABIDE vs ADHD-200
Side-by-side stats to understand how the two datasets differ.

In [ ]:
# ABIDE is larger and from more sites, which may help or hurt generalization.
# Both use CPAC preprocessing so the derivatives should be comparable.
import pandas as pd

adhd_pheno = load_phenotypic('../data/raw')
abide_pheno = load_phenotypic('../data/raw_abide')

rows = [
    {'Dataset': 'ADHD-200', 'Total': len(adhd_pheno),
     'Disorder': (adhd_pheno['label']==0).sum(),
     'Control':  (adhd_pheno['label']==1).sum(),
     'Task': 'ADHD classification'},
    {'Dataset': 'ABIDE',    'Total': len(abide_pheno),
     'Disorder': (abide_pheno['label']==0).sum(),
     'Control':  (abide_pheno['label']==1).sum(),
     'Task': 'Autism classification'},
]

df = pd.DataFrame(rows)
print(df.to_string(index=False))